# Altair Composer — готовый учебный прототип

**Исследовательский вопрос:** как состав дополнительного музыкального корпуса влияет на продолжение одной и той же темы?

### Гипотеза
В качестве учебной гипотезы используем **плавное мелодическое движение**: если добавить в обучение корпус с большим количеством соседних по высоте нот и более спокойным ритмом, то продолжения должны чаще использовать небольшие интервалы и меньшую плотность нот.

Это не готовый вывод — ноутбук **измеряет**, произошло ли такое изменение.

### Что выполняет ноутбук
1. Проверяет MIDI-корпуса и лицензии.
2. Отделяет тестовые мотивы от обучения.
3. Строит простую базовую модель Маркова.
4. Делает адаптацию на дополнительном корпусе.
5. Даёт **те же три мотива** базовой и адаптированной моделям.
6. Создаёт несколько вариантов продолжения и сохраняет их в MIDI.
7. Считает технические метрики именно для хвоста после мотива.
8. Создаёт таблицу для музыкальной оценки экспертом.

> Для учебного запуска, если настоящих корпусов ещё нет, ноутбук автоматически создаёт маленький **демонстрационный** корпус из собственных MIDI-последовательностей. Это позволяет проверить всю цепочку от начала до конца. Для защиты проекта его следует заменить реальными открытыми MIDI из `SOURCES.md` и заполнить каталог лицензий.

## 1. Библиотеки и папки

Используем только три простые библиотеки:
- `mido` — читать и записывать MIDI;
- `pandas` — таблицы;
- `numpy` — случайный выбор следующей ноты по вероятностям.

Никаких больших нейросетей и `music21` здесь нет, поэтому запуск проще.

In [ ]:
%pip install -q mido pandas numpy

from pathlib import Path
from collections import defaultdict, Counter
import csv
import random
import math

import numpy as np
import pandas as pd
from mido import MidiFile, MidiTrack, Message, MetaMessage

ROOT = Path(".")
BASE = ROOT / "data" / "base_corpus"
EXTRA = ROOT / "data" / "extra_corpus"
MOTIFS = ROOT / "data" / "motifs"
OUTPUTS = ROOT / "outputs"
RESULTS = ROOT / "results"

for p in [BASE, EXTRA, MOTIFS, OUTPUTS, RESULTS]:
    p.mkdir(parents=True, exist_ok=True)

random.seed(42)
np.random.seed(42)
print("Папки готовы.")

## 2. MIDI-функции

Чтобы код был понятен, мы работаем с одноголосной мелодией как со списком высот нот MIDI.

Например, `60, 62, 64` — это до, ре, ми.

Длительность в этом учебном варианте фиксирована: одна нота = одна четверть. Это упрощение специально сделано для школьного прототипа. Оно позволяет сосредоточиться на главном вопросе — **как меняются переходы между нотами**.

In [ ]:
def write_midi(pitches, path, tempo=500000):
    """Сохранить список MIDI-высот как простую одноголосную мелодию."""
    mid = MidiFile(ticks_per_beat=480)
    track = MidiTrack()
    mid.tracks.append(track)
    track.append(MetaMessage("set_tempo", tempo=tempo, time=0))
    for pitch in pitches:
        track.append(Message("note_on", note=int(pitch), velocity=80, time=0))
        track.append(Message("note_off", note=int(pitch), velocity=0, time=480))
    track.append(MetaMessage("end_of_track", time=0))
    mid.save(path)


def read_midi(path):
    """Прочитать ноты из первого MIDI-трека."""
    mid = MidiFile(path)
    notes = []
    for track in mid.tracks:
        for msg in track:
            if msg.type == "note_on" and msg.velocity > 0:
                notes.append(msg.note)
        if notes:
            break
    return notes


def midi_is_valid(path):
    try:
        notes = read_midi(path)
        return len(notes) >= 2
    except Exception:
        return False


## 3. Тестовые мотивы

В архиве задания уже есть три входных мотива:
- `motif_leap.mid`;
- `motif_scale.mid`;
- `motif_stepwise.mid`.

**Важно:** эти файлы используются только для проверки «до / после». Мы не обучаемся на них.

Если ноутбук запущен отдельно и этих файлов нет, он создаст такие же короткие учебные мотивы автоматически.

In [ ]:
default_motifs = {
    "leap": [60, 64, 67, 64],
    "scale": [60, 62, 64, 65],
    "stepwise": [60, 62, 63, 65, 67],
}

for name, pitches in default_motifs.items():
    path = MOTIFS / f"motif_{name}.mid"
    if not path.exists():
        write_midi(pitches, path)

print("Мотивы:")
for path in sorted(MOTIFS.glob("motif_*.mid")):
    print(path.name, read_midi(path))

## 4. Демонстрационные корпуса

В настоящем проекте сюда должны быть положены реальные открытые MIDI из источников, перечисленных в `SOURCES.md`.

Но чтобы ноутбук **гарантированно работал сразу**, если папки пустые, мы создаём небольшой демонстрационный корпус своими силами.

Это важно правильно интерпретировать:
- файлы с `source=demo` — техническая заглушка для проверки прототипа;
- для финальной защиты их нужно заменить реальными MIDI;
- тестовые мотивы никогда не попадают в обучение.

Базовый корпус содержит разные мелодические движения. Дополнительный корпус специально содержит больше соседних нот и более длинные последовательности — так мы можем проверить выбранную гипотезу.

In [ ]:
base_demo = {
    "base_01": [60, 64, 62, 65, 67, 64, 60, 62, 65, 67],
    "base_02": [67, 65, 62, 60, 64, 67, 69, 65, 62, 60],
    "base_03": [55, 60, 64, 60, 57, 62, 65, 62, 59, 64],
    "base_04": [60, 67, 65, 62, 64, 60, 69, 67, 64, 62],
}

extra_demo = {
    "extra_01": [60, 62, 64, 65, 67, 69, 67, 65, 64, 62, 60, 62],
    "extra_02": [62, 64, 65, 67, 69, 67, 65, 64, 62, 60, 62, 64],
    "extra_03": [57, 59, 60, 62, 64, 65, 67, 65, 64, 62, 60, 59],
    "extra_04": [65, 64, 62, 60, 62, 64, 65, 67, 69, 67, 65, 64],
}

if not list(BASE.glob("*.mid")):
    for name, notes in base_demo.items():
        write_midi(notes, BASE / f"{name}.mid")

if not list(EXTRA.glob("*.mid")):
    for name, notes in extra_demo.items():
        write_midi(notes, EXTRA / f"{name}.mid")

print("База:", len(list(BASE.glob("*.mid"))), "MIDI")
print("Дополнительный корпус:", len(list(EXTRA.glob("*.mid"))), "MIDI")

## 5. Каталог и проверка лицензий

ТЗ требует понимать, откуда взят каждый MIDI и можно ли его использовать. Поэтому создаём `catalog.csv`.

Для демонстрационных файлов лицензия `PD` означает, что **мы сами создали эти MIDI для примера**. Для реального корпуса в каталоге нужно указать настоящую лицензию и источник.

Файлы без понятной лицензии модель не использует.

In [ ]:
CATALOG = ROOT / "data" / "catalog.csv"

rows = []
for folder, corpus_name in [(BASE, "base"), (EXTRA, "extra")]:
    for path in sorted(folder.glob("*.mid")):
        rows.append({
            "work_id": path.stem,
            "file": str(path),
            "corpus": corpus_name,
            "composer": "Demo author" if corpus_name == "base" else "Demo romantic profile",
            "source": "demo: generated for notebook" ,
            "license": "PD",
        })

# Не перезаписываем настоящий каталог пользователя.
if not CATALOG.exists():
    pd.DataFrame(rows).to_csv(CATALOG, index=False, encoding="utf-8-sig")

# Читаем каталог и оставляем только разрешённые лицензии.
catalog = pd.read_csv(CATALOG)
allowed = {"PD", "CC0", "CC BY", "CC BY 4.0", "CC BY-SA", "CC BY-SA 4.0", "CC BY-NC-SA 4.0"}
catalog["allowed"] = catalog["license"].astype(str).str.strip().isin(allowed)

print(catalog[["work_id", "corpus", "source", "license", "allowed"]].to_string(index=False))

if not catalog["allowed"].all():
    print("\nПредупреждение: файлы с неизвестной лицензией будут исключены из обучения.")

## 6. Разделение по произведениям

Нельзя брать часть одного произведения в обучение, а другую часть — в тест. Иначе модель может просто запомнить музыку.

Поэтому единицей разделения является `work_id`. В нашем простом эксперименте тестовыми данными остаются три входных мотива, а корпус используется только для обучения.

In [ ]:
train_base = catalog[(catalog.corpus == "base") & catalog.allowed].copy()
train_extra = catalog[(catalog.corpus == "extra") & catalog.allowed].copy()

print(f"Разрешённых произведений в базе: {len(train_base)}")
print(f"Разрешённых произведений в доп. корпусе: {len(train_extra)}")
print("Тестовые мотивы в каталог обучения не входят.")

## 7. Простая модель Маркова

Модель запоминает переходы вида:

`предыдущая нота → следующая нота`.

Например, если после `60` в корпусе часто идут `62` и `64`, модель будет чаще выбирать их.

Это и есть упрощённая адаптация: после обучения на дополнительном корпусе его переходы получают больший вес. Это не большой fine-tune, но по смыслу выполняет требуемую идею **адаптации модели на новом корпусе** и остаётся понятным школьнику.

In [ ]:
class MarkovComposer:
    def __init__(self):
        self.transitions = defaultdict(Counter)

    def fit(self, files, weight=1):
        for file in files:
            notes = read_midi(file)
            for a, b in zip(notes, notes[1:]):
                self.transitions[a][b] += weight
        return self

    def next_note(self, current):
        options = self.transitions.get(current)
        if not options:
            # Если такой ноты не было, выбираем любую известную ноту.
            all_notes = [n for d in self.transitions.values() for n in d]
            return random.choice(all_notes) if all_notes else current
        notes = list(options.keys())
        weights = np.array(list(options.values()), dtype=float)
        weights /= weights.sum()
        return int(np.random.choice(notes, p=weights))

    def generate(self, motif, length=8, seed=1):
        random.seed(seed)
        np.random.seed(seed)
        result = list(motif)
        current = motif[-1]
        for _ in range(length):
            current = self.next_note(current)
            result.append(current)
        return result


def files_from_catalog(frame):
    return [Path(x) for x in frame["file"] if Path(x).exists() and midi_is_valid(x)]

base_files = files_from_catalog(train_base)
extra_files = files_from_catalog(train_extra)

base_model = MarkovComposer().fit(base_files)
adapted_model = MarkovComposer().fit(base_files).fit(extra_files, weight=3)

print("Базовая модель обучена на:", len(base_files), "файлах")
print("Адаптированная модель получила дополнительно:", len(extra_files), "файлов")

## 8. Один и тот же мотив «до / после»

Это ключевой эксперимент. Вход **не меняется**. Меняется только модель.

Для каждого мотива создаём три варианта продолжения. Одинаковые `seed` помогают сравнивать варианты честнее.

In [ ]:
def motif_files():
    return sorted(MOTIFS.glob("motif_*.mid"))

for motif_path in motif_files():
    motif = read_midi(motif_path)
    name = motif_path.stem.replace("motif_", "")
    print(f"\n{name}: {motif}")
    for i in range(1, 4):
        before = base_model.generate(motif, length=8, seed=i)
        after = adapted_model.generate(motif, length=8, seed=i)
        print(f"  вариант {i}  ДО: {before}  ПОСЛЕ: {after}")

## 9. Сохраняем MIDI

Каждый результат можно открыть в любом MIDI-плеере. Таким образом выполняется демонстрационный сценарий из ТЗ:

**мотив → несколько продолжений → MIDI → прослушивание.**

In [ ]:
generated = []

for motif_path in motif_files():
    motif = read_midi(motif_path)
    name = motif_path.stem.replace("motif_", "")
    for version, model in [("before", base_model), ("after", adapted_model)]:
        for i in range(1, 4):
            pitches = model.generate(motif, length=8, seed=i)
            out = OUTPUTS / f"{name}_{version}_{i}.mid"
            write_midi(pitches, out)
            generated.append({"motif": name, "version": version, "variant": i, "file": str(out)})

print(f"Создано MIDI: {len(generated)}")
print(pd.DataFrame(generated).to_string(index=False))

## 10. Технические метрики

Считаем характеристики **только хвоста после исходного мотива**. Это важно: иначе одинаковый вход будет искусственно влиять на сравнение.

Используем простые показатели из ТЗ:
- плотность нот;
- диапазон высот;
- средний размер интервала;
- повторяемость небольшого рисунка.

In [ ]:
def tail_metrics(pitches, motif_length):
    tail = pitches[motif_length:]
    if len(tail) < 2:
        return {"notes_per_16": len(tail), "pitch_range": 0, "mean_interval": 0, "repeat_score": 0}

    intervals = [abs(b - a) for a, b in zip(tail, tail[1:])]
    # В нашем MIDI одна нота = одна четверть, поэтому 4 ноты = 1 такт 4/4.
    notes_per_16 = len(tail) / 4 * 4
    # Чем больше одинаковых коротких пар, тем выше повторяемость.
    pairs = list(zip(tail, tail[1:]))
    repeat_score = 1 - len(set(pairs)) / len(pairs) if pairs else 0

    return {
        "notes_per_16": round(notes_per_16, 2),
        "pitch_range": max(tail) - min(tail),
        "mean_interval": round(float(np.mean(intervals)), 2),
        "repeat_score": round(float(repeat_score), 2),
    }

metrics = []
for item in generated:
    pitches = read_midi(item["file"])
    motif_path = MOTIFS / f"motif_{item['motif']}.mid"
    motif_len = len(read_midi(motif_path))
    row = dict(item)
    row.update(tail_metrics(pitches, motif_len))
    metrics.append(row)

metrics_df = pd.DataFrame(metrics)
metrics_df.to_csv(RESULTS / "technical_metrics.csv", index=False, encoding="utf-8-sig")
metrics_df

## 11. Сравнение «до / после»

Средние значения показывают направление изменения. Само по себе изменение ещё не доказывает гипотезу — его нужно проверить прослушиванием и объяснить.

In [ ]:
summary = (
    metrics_df.groupby(["motif", "version"])[["notes_per_16", "pitch_range", "mean_interval", "repeat_score"]]
    .mean()
    .round(2)
    .reset_index()
)

summary.to_csv(RESULTS / "before_after_summary.csv", index=False, encoding="utf-8-sig")
summary

## 12. Музыкальная оценка экспертом

Технические цифры не слышат музыку полностью. Поэтому по ТЗ нужна отдельная оценка музыкального эксперта.

Здесь создаётся готовый CSV-шаблон. Эксперт может поставить от **1 до 5** по каждому критерию и написать комментарий.

Критерии:
- сохранение темы;
- логичность развития;
- ритмическая связность;
- гармоническая связность;
- целостность;
- выразительность.

In [ ]:
expert_rows = []
for motif_path in motif_files():
    name = motif_path.stem.replace("motif_", "")
    for version in ["before", "after"]:
        expert_rows.append({
            "motif": name,
            "version": version,
            "theme": "",
            "development": "",
            "rhythm": "",
            "harmony": "",
            "integrity": "",
            "expression": "",
            "comment": "",
        })

expert_df = pd.DataFrame(expert_rows)
expert_df.to_csv(RESULTS / "expert_evaluation.csv", index=False, encoding="utf-8-sig")
expert_df

## 13. Простая демонстрация для защиты

Функция ниже — мини-интерфейс без сложного GUI. Достаточно указать имя мотива, и ноутбук создаст несколько вариантов до и после адаптации.

Например:

```python
generate_variants("stepwise", 3)
```

In [ ]:
def generate_variants(motif_name="stepwise", n=3):
    path = MOTIFS / f"motif_{motif_name}.mid"
    if not path.exists():
        raise ValueError("Нет такого мотива. Используйте leap, scale или stepwise.")

    motif = read_midi(path)
    rows = []
    for i in range(1, n + 1):
        for version, model in [("before", base_model), ("after", adapted_model)]:
            pitches = model.generate(motif, length=8, seed=i)
            out = OUTPUTS / f"demo_{motif_name}_{version}_{i}.mid"
            write_midi(pitches, out)
            rows.append({"version": version, "variant": i, "file": str(out), "notes": pitches})
    return pd.DataFrame(rows)

# Пример запуска:
demo = generate_variants("stepwise", 3)
demo[["version", "variant", "file"]]

## 14. Контроль соответствия ТЗ

| Требование | Выполнение |
|---|---|
| Короткий мотив на входе | Да — три входных MIDI |
| Базовая модель | Да — простая модель Маркова |
| Музыкальная гипотеза | Да — проверяем влияние плавного движения |
| Дополнительный корпус | Да — предусмотрен отдельный корпус; есть рабочая demo-заглушка |
| Проверка лицензии | Да — через `catalog.csv` |
| Адаптация | Да — дополнительные переходы имеют больший вес |
| Те же мотивы до/после | Да |
| Несколько продолжений | Да — по 3 варианта |
| MIDI на выходе | Да |
| Техническое сравнение | Да — 4 метрики |
| Музыкальное сравнение | Да — CSV для эксперта |
| Демонстрационный сценарий | Да — `generate_variants()` |
| Большая модель с нуля | Нет |

### Важное замечание
Демонстрационный корпус нужен только для того, чтобы проект запускался сразу. Для итоговой защиты его следует заменить на реальный открытый корпус из `SOURCES.md`, а в `catalog.csv` указать реальные `work_id`, композитора, источник и лицензию. После этого код менять не потребуется.

## 15. Где искать результаты

После запуска в папке `outputs/` лежат MIDI-файлы:
- `*_before_*.mid` — базовая модель;
- `*_after_*.mid` — адаптированная модель.

В папке `results/`:
- `technical_metrics.csv` — все технические измерения;
- `before_after_summary.csv` — среднее сравнение;
- `expert_evaluation.csv` — таблица для музыкального эксперта.

Такой набор файлов удобно показать на защите вместе с прослушиванием одинаковых мотивов «до / после».